# ElevanceSkills Technology Private Limited
## Data Visualization Internship — All 6 Tasks
### Intern: Adinath Chavan

---

| Detail | Info |
|---|---|
| **Dataset** | Job Descriptions Dataset (Kaggle) |
| **Libraries Used** | NumPy and Pandas ONLY |
| **Visualization Tool** | Tableau |
| **Tasks** | 6 Tasks |
| **Organization** | ElevanceSkills Technology Private Limited |

> ✅ Python (NumPy + Pandas) is used for **data filtering and preparation only**.  
> ✅ Each task exports a **filtered CSV file** which is then loaded into **Tableau** for visualization.

---

## 📚 Step 1 — Import Libraries

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import warnings
warnings.filterwarnings('ignore')

print('✅ NumPy  version :', np.__version__)
print('✅ Pandas version :', pd.__version__)
print('✅ Libraries imported successfully!')

✅ NumPy  version : 2.4.4
✅ Pandas version : 3.0.2
✅ Libraries imported successfully!


## 📂 Step 2 — Load and Clean Dataset

> ⚠️ Make sure **job_descriptions.csv** is in the same folder as this notebook.

In [6]:
# Load dataset
df = pd.read_csv(r'C:\Users\DELL\Documents\python file\job_descriptions.csv')
# Strip whitespace from column names
df.columns = df.columns.str.strip()

print('✅ Dataset loaded successfully!')
print(f'   Rows    : {df.shape[0]:,}')
print(f'   Columns : {df.shape[1]}')
print(f'\nColumn Names:')
for col in df.columns:
    print(f'   - {col}')

✅ Dataset loaded successfully!
   Rows    : 1,615,940
   Columns : 23

Column Names:
   - Job Id
   - Experience
   - Qualifications
   - Salary Range
   - location
   - Country
   - latitude
   - longitude
   - Work Type
   - Company Size
   - Job Posting Date
   - Preference
   - Contact Person
   - Contact
   - Job Title
   - Role
   - Job Portal
   - Job Description
   - Benefits
   - skills
   - Responsibilities
   - Company
   - Company Profile


In [7]:
# ── DATA CLEANING ─────────────────────────────────────────────────

# Clean Salary — remove $, commas → convert to float
if df['Salary'].dtype == object:
    df['Salary'] = df['Salary'].str.replace('[\$,]', '', regex=True)
    df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

# Clean Company Size — remove commas → convert to float
if df['Company Size'].dtype == object:
    df['Company Size'] = df['Company Size'].str.replace(',', '', regex=False)
    df['Company Size'] = pd.to_numeric(df['Company Size'], errors='coerce')

# Clean Experience — extract first number (e.g. '3 years' → 3)
if df['Experience'].dtype == object:
    df['Experience'] = df['Experience'].str.extract(r'(\d+)').astype(float)

# Parse Posted Date to datetime
if 'Posted Date' in df.columns:
    df['Posted Date'] = pd.to_datetime(df['Posted Date'], errors='coerce')

# Strip whitespace from all string columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

print('✅ Data cleaning complete!')
print(f'\nData Types after Cleaning:')
print(df[['Salary','Company Size','Experience']].dtypes)

KeyError: 'Salary'

In [ ]:
# Preview first 5 rows
print('Dataset Preview (first 5 rows):')
df.head()

In [ ]:
# Statistical summary using NumPy and Pandas
print('Statistical Summary of Key Columns:')
summary = pd.DataFrame({
    'Column'  : ['Salary', 'Company Size', 'Experience'],
    'Min'     : [np.nanmin(df['Salary']),       np.nanmin(df['Company Size']),  np.nanmin(df['Experience'])],
    'Max'     : [np.nanmax(df['Salary']),       np.nanmax(df['Company Size']),  np.nanmax(df['Experience'])],
    'Mean'    : [np.nanmean(df['Salary']),      np.nanmean(df['Company Size']), np.nanmean(df['Experience'])],
    'Median'  : [np.nanmedian(df['Salary']),    np.nanmedian(df['Company Size']),np.nanmedian(df['Experience'])],
    'Std Dev' : [np.nanstd(df['Salary']),       np.nanstd(df['Company Size']),  np.nanstd(df['Experience'])],
    'Nulls'   : [df['Salary'].isna().sum(),     df['Company Size'].isna().sum(),df['Experience'].isna().sum()]
})
summary = summary.set_index('Column').round(2)
print(summary)

## ⏰ Helper — IST Time Check
Tasks 2, 3, 5, 6 are restricted to specific IST time windows.

In [ ]:
def check_ist_time(start_hour, end_hour):
    """
    Returns True if current IST time is within start_hour to end_hour.
    IST = UTC + 5:30
    """
    IST = timezone(timedelta(hours=5, minutes=30))
    now_ist = datetime.now(IST)
    in_range = start_hour <= now_ist.hour < end_hour
    print(f'   Current IST Time : {now_ist.strftime("%I:%M %p")}')
    print(f'   Allowed Window   : {start_hour}:00 – {end_hour}:00')
    print(f'   Status           : {"✅ WITHIN" if in_range else "⚠️  OUTSIDE"} allowed window')
    return in_range

print('✅ IST time check function ready!')

---
# 📊 TASK 1 — Preference vs Work Type
**Chart Type:** Bar Chart *(built in Tableau)*

| Filter | Condition |
|---|---|
| Work Type | = Intern |
| Company Size | < 50,000 |
| Salary | > $9,000 |
| Sort | Descending by Count |

In [ ]:
print('=' * 55)
print('  TASK 1 — Preference vs Work Type')
print('=' * 55)

# ── Apply Filters ─────────────────────────────────────────
t1 = df[
    (df['Work Type'] == 'Intern') &
    (df['Company Size'] < 50000) &
    (df['Salary'] > 9000)
].copy()

print(f'\n✅ Total records after filtering : {len(t1):,}')

# ── Analysis using Pandas & NumPy ─────────────────────────
counts = t1['Preference'].value_counts().sort_values(ascending=False)
total  = counts.sum()

result = pd.DataFrame({
    'Preference'   : counts.index,
    'Count'        : counts.values,
    'Percentage %' : np.round((counts.values / total) * 100, 2)
})

print(f'\n📊 Preference Distribution (Sorted Descending):')
print(result.to_string(index=False))
print(f'\n   Total Records : {total:,}')
print(f'   Top Preference: {counts.index[0]} ({counts.values[0]:,} records)')

# ── Export filtered CSV for Tableau ───────────────────────
t1.to_csv('task1_filtered.csv', index=False)
print(f'\n✅ Exported → task1_filtered.csv  (for Tableau)')

---
# 📊 TASK 2 — Company Size vs Company Name
**Chart Type:** Scatter Plot *(built in Tableau)*

| Filter | Condition |
|---|---|
| Job Title | = Mechanical Engineer |
| Company Size | < 50,000 |
| Experience | > 5 years |
| Salary | > $50,000 |
| Work Type | Full-Time or Part-Time |
| Preference | = Male |
| Country | Asian countries excluding starting with 'I' |
| Job Portal | = Idealist |
| Company Name | At least 2 vowels |
| Time | 3 PM – 5 PM IST only |

In [ ]:
print('=' * 55)
print('  TASK 2 — Company Size vs Company Name')
print('=' * 55)

# ── IST Time Check ────────────────────────────────────────
print('\n⏰ IST Time Check:')
check_ist_time(15, 17)

# ── Asian Countries excluding starting with I ─────────────
asian_countries = np.array([
    'China', 'Japan', 'South Korea', 'Thailand', 'Vietnam',
    'Malaysia', 'Singapore', 'Philippines', 'Bangladesh',
    'Nepal', 'Sri Lanka', 'Myanmar', 'Cambodia', 'Laos',
    'Mongolia', 'Kazakhstan', 'Uzbekistan', 'Afghanistan',
    'Pakistan', 'Saudi Arabia', 'UAE', 'Qatar', 'Kuwait',
    'Bahrain', 'Oman', 'Jordan', 'Lebanon', 'Syria'
])
# Exclude countries starting with 'I'
asian_countries = asian_countries[~np.char.startswith(asian_countries.astype(str), 'I')]
print(f'\n   Asian countries used (excl. I): {list(asian_countries)}')

# ── Helper: company has 2+ vowels ─────────────────────────
def has_two_vowels(name):
    if pd.isna(name): return False
    return sum(1 for ch in str(name).lower() if ch in 'aeiou') >= 2

# ── Apply Filters ─────────────────────────────────────────
t2 = df[
    (df['Job Title'] == 'Mechanical Engineer') &
    (df['Company Size'] < 50000) &
    (df['Experience'] > 5) &
    (df['Salary'] > 50000) &
    (df['Work Type'].isin(['Full-Time', 'Part-Time'])) &
    (df['Preference'] == 'Male') &
    (df['Country'].isin(asian_countries)) &
    (df['Job Portal'] == 'Idealist') &
    (df['Company'].apply(has_two_vowels))
].copy()

print(f'\n✅ Total records after filtering : {len(t2):,}')

# ── Analysis ──────────────────────────────────────────────
if not t2.empty:
    print(f'\n📊 Company Size Statistics:')
    print(f'   Min Company Size : {np.nanmin(t2["Company Size"]):,.0f}')
    print(f'   Max Company Size : {np.nanmax(t2["Company Size"]):,.0f}')
    print(f'   Mean Company Size: {np.nanmean(t2["Company Size"]):,.0f}')
    print(f'   Min Salary       : ${np.nanmin(t2["Salary"]):,.0f}')
    print(f'   Max Salary       : ${np.nanmax(t2["Salary"]):,.0f}')
    print(f'\n📋 Filtered Records Preview:')
    print(t2[['Company', 'Company Size', 'Salary', 'Country', 'Work Type']].to_string(index=False))
else:
    print('\n⚠️  No records match all Task 2 filters.')

# ── Export for Tableau ────────────────────────────────────
t2.to_csv('task2_filtered.csv', index=False)
print(f'\n✅ Exported → task2_filtered.csv  (for Tableau)')

---
# 📊 TASK 3 — Work Type Salary Distribution
**Chart Type:** Box & Whisker Plot *(built in Tableau)*

| Filter | Condition |
|---|---|
| Work Type | = Intern |
| Latitude | < 10 |
| Company Size | < 50,000 |
| Salary | > $8,000 |
| Job Title | Single word, fewer than 10 characters |
| Experience | Even number |
| Posted Date | Between 2021 and 2023 |
| Contact Person | Name contains at least one 'e' |
| Time | 3 PM – 5 PM IST only |

In [ ]:
print('=' * 55)
print('  TASK 3 — Salary Distribution (Box & Whisker)')
print('=' * 55)

# ── IST Time Check ────────────────────────────────────────
print('\n⏰ IST Time Check:')
check_ist_time(15, 17)

# ── Helper Functions ──────────────────────────────────────
def is_single_word_under10(title):
    if pd.isna(title): return False
    t = str(title).strip()
    return (len(t.split()) == 1) and (len(t) < 10)

def contact_has_e(name):
    if pd.isna(name): return False
    return 'e' in str(name).lower()

# ── Apply Filters ─────────────────────────────────────────
t3 = df[
    (df['Work Type'] == 'Intern') &
    (df['Latitude'] < 10) &
    (df['Company Size'] < 50000) &
    (df['Salary'] > 8000) &
    (df['Job Title'].apply(is_single_word_under10)) &
    (df['Experience'] % 2 == 0) &
    (df['Posted Date'].dt.year.between(2021, 2023)) &
    (df['Contact Person'].apply(contact_has_e))
].copy()

print(f'\n✅ Total records after filtering : {len(t3):,}')

# ── Box Plot Statistics using NumPy ───────────────────────
if not t3.empty:
    salary_arr = t3['Salary'].dropna().values
    q1  = np.percentile(salary_arr, 25)
    q2  = np.percentile(salary_arr, 50)
    q3  = np.percentile(salary_arr, 75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outliers    = salary_arr[(salary_arr < lower_fence) | (salary_arr > upper_fence)]

    print(f'\n📊 Box & Whisker Statistics (Salary):')
    print(f'   Minimum      : ${np.min(salary_arr):>12,.2f}')
    print(f'   Q1  (25%)    : ${q1:>12,.2f}')
    print(f'   Median (50%) : ${q2:>12,.2f}')
    print(f'   Q3  (75%)    : ${q3:>12,.2f}')
    print(f'   Maximum      : ${np.max(salary_arr):>12,.2f}')
    print(f'   IQR          : ${iqr:>12,.2f}')
    print(f'   Lower Fence  : ${lower_fence:>12,.2f}')
    print(f'   Upper Fence  : ${upper_fence:>12,.2f}')
    print(f'   Mean         : ${np.mean(salary_arr):>12,.2f}')
    print(f'   Std Dev      : ${np.std(salary_arr):>12,.2f}')
    print(f'   Outliers     : {len(outliers)} records')
    if len(outliers) > 0:
        print(f'   Outlier Values: {np.sort(outliers)}')
else:
    print('\n⚠️  No records match all Task 3 filters.')

# ── Export for Tableau ────────────────────────────────────
t3.to_csv('task3_filtered.csv', index=False)
print(f'\n✅ Exported → task3_filtered.csv  (for Tableau)')

---
# 📊 TASK 4 — India vs Germany Job Comparison
**Chart Type:** Stacked Bar Chart *(built in Tableau)*

| Filter | Condition |
|---|---|
| Country | India or Germany |
| Qualification | = B.Tech |
| Work Type | = Full-Time |
| Experience | > 2 years |
| Job Title | Data Scientist / Art Teacher / Aerospace Engineer |
| Salary | > $10,000 |
| Job Portal | = Indeed |
| Company Name | More than 8 characters |
| Location | Not empty |

In [ ]:
print('=' * 55)
print('  TASK 4 — India vs Germany Job Comparison')
print('=' * 55)

target_roles = ['Data Scientist', 'Art Teacher', 'Aerospace Engineer']

# ── Apply Filters ─────────────────────────────────────────
t4 = df[
    (df['Country'].isin(['India', 'Germany'])) &
    (df['Qualification'] == 'B.Tech') &
    (df['Work Type'] == 'Full-Time') &
    (df['Experience'] > 2) &
    (df['Job Title'].isin(target_roles)) &
    (df['Salary'] > 10000) &
    (df['Job Portal'] == 'Indeed') &
    (df['Company'].str.len() > 8) &
    (df['Location'].notna()) &
    (df['Location'] != '')
].copy()

print(f'\n✅ Total records after filtering : {len(t4):,}')

# ── Analysis using Pandas & NumPy ─────────────────────────
pivot = t4.groupby(['Job Title', 'Country']).size().unstack(fill_value=0)
for col in ['India', 'Germany']:
    if col not in pivot.columns:
        pivot[col] = 0

pivot['Total']         = pivot['India'] + pivot['Germany']
pivot['India %']       = np.round((pivot['India']   / pivot['Total'].replace(0,np.nan)) * 100, 1)
pivot['Germany %']     = np.round((pivot['Germany'] / pivot['Total'].replace(0,np.nan)) * 100, 1)

print(f'\n📊 Job Postings — India vs Germany:')
print(pivot.to_string())

print(f'\n📊 Country-wise Totals:')
country_totals = t4['Country'].value_counts()
print(f'   India   : {country_totals.get("India",   0):,} postings')
print(f'   Germany : {country_totals.get("Germany", 0):,} postings')

print(f'\n📊 Salary Statistics by Country (using NumPy):')
for country in ['India', 'Germany']:
    sal = t4[t4['Country'] == country]['Salary'].dropna().values
    if len(sal) > 0:
        print(f'   {country}:')
        print(f'      Mean   : ${np.mean(sal):,.2f}')
        print(f'      Median : ${np.median(sal):,.2f}')
        print(f'      Min    : ${np.min(sal):,.2f}')
        print(f'      Max    : ${np.max(sal):,.2f}')

# ── Export for Tableau ────────────────────────────────────
t4.to_csv('task4_filtered.csv', index=False)
print(f'\n✅ Exported → task4_filtered.csv  (for Tableau)')

---
# 📊 TASK 5 — Top 10 Companies
**Chart Type:** Tree Map *(built in Tableau)*

| Filter | Condition |
|---|---|
| Role | = Data Engineer |
| Job Title | = Data Scientist |
| Country | Exclude Asian countries & countries starting with 'C' |
| Company Size | ≥ 10,000 |
| Qualification | = B.Tech |
| Preference | = Female |
| Job Portal | = LinkedIn |
| Posted Date | 01/01/2023 – 06/01/2023 |
| Contact Person | Name ends with a vowel |
| Time | 3 PM – 5 PM IST only |

In [8]:
print('=' * 55)
print('  TASK 5 — Top 10 Companies (Tree Map)')
print('=' * 55)

# ── IST Time Check ────────────────────────────────────────
print('\n⏰ IST Time Check:')
check_ist_time(15, 17)

# ── Asian Countries List (full) ───────────────────────────
asian_countries_full = np.array([
    'China', 'India', 'Japan', 'South Korea', 'Thailand', 'Vietnam',
    'Malaysia', 'Singapore', 'Philippines', 'Bangladesh', 'Nepal',
    'Sri Lanka', 'Indonesia', 'Iran', 'Iraq', 'Myanmar', 'Cambodia',
    'Laos', 'Mongolia', 'Kazakhstan', 'Uzbekistan', 'Afghanistan',
    'Pakistan', 'Saudi Arabia', 'UAE', 'Qatar', 'Kuwait',
    'Bahrain', 'Oman', 'Jordan', 'Lebanon', 'Syria'
])

# ── Helper: contact person ends with vowel ────────────────
def ends_with_vowel(name):
    if pd.isna(name): return False
    return str(name).strip()[-1].lower() in 'aeiou'

# ── Apply Filters ─────────────────────────────────────────
t5 = df[
    (df['Role'] == 'Data Engineer') &
    (df['Job Title'] == 'Data Scientist') &
    (~df['Country'].isin(asian_countries_full)) &
    (~df['Country'].str.startswith('C', na=False)) &
    (df['Company Size'] >= 10000) &
    (df['Qualification'] == 'B.Tech') &
    (df['Preference'] == 'Female') &
    (df['Job Portal'] == 'LinkedIn') &
    (df['Posted Date'] >= pd.Timestamp('2023-01-01')) &
    (df['Posted Date'] <= pd.Timestamp('2023-06-01')) &
    (df['Contact Person'].apply(ends_with_vowel))
].copy()

print(f'\n✅ Total records after filtering : {len(t5):,}')

# ── Top 10 Companies using Pandas & NumPy ────────────────
top10 = t5['Company'].value_counts().head(10)
top10_df = pd.DataFrame({
    'Rank'        : np.arange(1, len(top10)+1),
    'Company'     : top10.index,
    'Postings'    : top10.values,
    'Share %'     : np.round((top10.values / top10.values.sum()) * 100, 2)
})

print(f'\n📊 Top 10 Companies (Tree Map Data):')
print(top10_df.to_string(index=False))

# ── Export Top 10 for Tableau ─────────────────────────────
top10_df.to_csv('task5_top10.csv', index=False)
t5.to_csv('task5_filtered.csv', index=False)
print(f'\n✅ Exported → task5_top10.csv     (for Tableau Tree Map)')
print(f'✅ Exported → task5_filtered.csv  (full filtered data)')

  TASK 5 — Top 10 Companies (Tree Map)

⏰ IST Time Check:


NameError: name 'check_ist_time' is not defined

---
# 📊 TASK 6 — Qualification Drilldown Map
**Chart Type:** Geographic Map with Drilldown *(built in Tableau)*

| Filter | Condition |
|---|---|
| Country | African countries only |
| Qualification | B.Tech, M.Tech, or PhD |
| Work Type | = Full-Time |
| Job Title | Starts with 'D' |
| Preference | = Male |
| Company Size | > 80,000 |
| Salary | > $20,000 |
| Contact Person | Name starts with 'A' |
| Job Portal | = Indeed |
| Time | 3 PM – 6 PM IST only |

In [ ]:
print('=' * 55)
print('  TASK 6 — Qualification Drilldown Map')
print('=' * 55)

# ── IST Time Check ────────────────────────────────────────
print('\n⏰ IST Time Check:')
check_ist_time(15, 18)

african_countries = np.array([
    'Nigeria', 'Ethiopia', 'Egypt', 'DR Congo', 'Tanzania',
    'South Africa', 'Kenya', 'Uganda', 'Algeria', 'Sudan',
    'Morocco', 'Angola', 'Mozambique', 'Ghana', 'Madagascar',
    'Cameroon', 'Ivory Coast', 'Niger', 'Burkina Faso', 'Mali',
    'Malawi', 'Zambia', 'Senegal', 'Chad', 'Somalia',
    'Zimbabwe', 'Guinea', 'Rwanda', 'Benin', 'Burundi',
    'Tunisia', 'South Sudan', 'Togo', 'Sierra Leone', 'Libya',
    'Congo', 'Liberia', 'Central African Republic', 'Mauritania',
    'Eritrea', 'Namibia', 'Gambia', 'Botswana', 'Gabon',
    'Lesotho', 'Guinea-Bissau', 'Equatorial Guinea', 'Mauritius',
    'Eswatini', 'Djibouti', 'Comoros', 'Cape Verde', 'Sao Tome'
])

def starts_with_A(name):
    if pd.isna(name): return False
    return str(name).strip().startswith('A')

# ── Apply Filters ─────────────────────────────────────────
t6 = df[
    (df['Country'].isin(african_countries)) &
    (df['Qualification'].isin(['B.Tech', 'M.Tech', 'PhD'])) &
    (df['Work Type'] == 'Full-Time') &
    (df['Job Title'].str.startswith('D', na=False)) &
    (df['Preference'] == 'Male') &
    (df['Company Size'] > 80000) &
    (df['Salary'] > 20000) &
    (df['Contact Person'].apply(starts_with_A)) &
    (df['Job Portal'] == 'Indeed')
].copy()

print(f'\n✅ Total records after filtering : {len(t6):,}')

# ── Analysis using Pandas & NumPy ─────────────────────────
if not t6.empty:
    print(f'\n📊 Qualification Breakdown:')
    qual_counts = t6['Qualification'].value_counts()
    qual_df = pd.DataFrame({
        'Qualification': qual_counts.index,
        'Count'        : qual_counts.values,
        'Percentage %' : np.round((qual_counts.values / qual_counts.sum()) * 100, 2)
    })
    print(qual_df.to_string(index=False))

    print(f'\n📊 Country-wise Distribution:')
    country_counts = t6['Country'].value_counts()
    print(country_counts.to_string())

    print(f'\n📊 Salary Statistics by Qualification (NumPy):')
    for qual in ['B.Tech', 'M.Tech', 'PhD']:
        sal = t6[t6['Qualification'] == qual]['Salary'].dropna().values
        if len(sal) > 0:
            print(f'   {qual}:')
            print(f'      Mean   : ${np.mean(sal):,.2f}')
            print(f'      Median : ${np.median(sal):,.2f}')
            print(f'      Min    : ${np.min(sal):,.2f}')
            print(f'      Max    : ${np.max(sal):,.2f}')

    print(f'\n📊 Latitude / Longitude Range (for Map):')
    print(f'   Latitude  : {np.nanmin(t6["Latitude"]):,.4f}  to  {np.nanmax(t6["Latitude"]):,.4f}')
    print(f'   Longitude : {np.nanmin(t6["Longitude"]):,.4f}  to  {np.nanmax(t6["Longitude"]):,.4f}')
else:
    print('\n⚠️  No records match all Task 6 filters.')

# ── Export for Tableau ────────────────────────────────────
t6.to_csv('task6_filtered.csv', index=False)
print(f'\n✅ Exported → task6_filtered.csv  (for Tableau Map)')

---
## 📁 All Exported CSV Files Summary

In [ ]:
import os

files = [
    ('task1_filtered.csv',  'Task 1 — Preference vs Work Type'),
    ('task2_filtered.csv',  'Task 2 — Company Size vs Company Name'),
    ('task3_filtered.csv',  'Task 3 — Salary Distribution'),
    ('task4_filtered.csv',  'Task 4 — India vs Germany'),
    ('task5_filtered.csv',  'Task 5 — Top 10 Companies (full)'),
    ('task5_top10.csv',     'Task 5 — Top 10 Companies (summary)'),
    ('task6_filtered.csv',  'Task 6 — Qualification Map'),
]

print('=' * 60)
print('  EXPORTED FILES — Load these into Tableau')
print('=' * 60)
for fname, desc in files:
    exists = os.path.exists(fname)
    size   = os.path.getsize(fname) if exists else 0
    rows   = sum(1 for _ in open(fname)) - 1 if exists else 0
    status = '✅' if exists else '❌'
    print(f' {status} {fname}')
    print(f'    → {desc}')
    if exists:
        print(f'    → {rows:,} rows | {size:,} bytes')
    print()

---
---
# 🟦 TABLEAU GUIDE — How to Build All 6 Charts

> Load each filtered CSV into Tableau and follow the steps below.

---

## ✅ How to Connect CSV to Tableau
1. Open **Tableau Desktop** or **Tableau Public** (free)
2. Click **Connect → Text File**
3. Select the relevant `taskX_filtered.csv` file
4. Click **Sheet 1** tab at the bottom to start building

---

## 📊 Task 1 — Bar Chart (Preference vs Work Type)
**File to load:** `task1_filtered.csv`

**Steps:**
1. Drag **Preference** → **Columns** shelf
2. Drag **Preference** → **Rows** shelf → Right-click → **Measure → Count**
3. In Marks panel → Change to **Bar**
4. Drag **Preference** → **Color** card
5. Click **Sort Descending** button (toolbar)
6. Drag **CNT(Preference)** → **Label** card → Check **Show Labels**
7. Title: `Preference vs Work Type | Intern | Company Size < 50,000 | Salary > $9,000`
8. Export: **Worksheet → Export → Image → task1_bar_chart.png**

---

## 📊 Task 2 — Scatter Plot (Company Size vs Company Name)
**File to load:** `task2_filtered.csv`

**Steps:**
1. Drag **Company** → **Columns** shelf
2. Drag **Company Size** → **Rows** shelf (it becomes AVG — right-click → **Dimension**)
3. In Marks panel → Change to **Circle**
4. Drag **Salary** → **Color** card (gradient color)
5. Drag **Country**, **Salary**, **Experience** → **Tooltip** card
6. Right-click X-axis → **Sort → Ascending by Company Size**
7. Title: `Company Size vs Company Name | Mechanical Engineer | Idealist | 3–5 PM IST`
8. Add text annotation: Right-click → **Annotate → Area** → type `Visible: 3–5 PM IST only`
9. Export: **task2_scatter_plot.png**

---

## 📊 Task 3 — Box & Whisker Plot (Salary Distribution)
**File to load:** `task3_filtered.csv`

**Steps:**
1. Drag **Work Type** → **Columns** shelf
2. Drag **Salary** → **Rows** shelf
3. Go to **Analysis menu → Aggregate Measures** → **Uncheck it**
4. In Marks panel → Change to **Circle**
5. Click **Analytics** tab (left panel) → Drag **Box Plot** onto the chart
6. Right-click box → **Format** → Set box color to **Light Blue**, median line to **Orange**
7. Drag **Salary**, **Job Title**, **Company** → **Tooltip** card
8. Title: `Salary Distribution — Intern Roles | Latitude < 10 | 2021–2023 | 3–5 PM IST`
9. Export: **task3_boxplot.png**

---

## 📊 Task 4 — Grouped Bar Chart (India vs Germany)
**File to load:** `task4_filtered.csv`

**Steps:**
1. Drag **Job Title** → **Columns** shelf
2. Drag **Country** → **Columns** shelf (next to Job Title — creates side-by-side bars)
3. Drag **Number of Records** → **Rows** shelf
4. In Marks panel → Change to **Bar**
5. Drag **Country** → **Color** card
6. Click **Color → Edit Colors**:
   - India = **Orange (#FF6B35)**
   - Germany = **Dark Blue (#1B4F8A)**
7. Drag **Number of Records** → **Label** card → Check **Show Labels**
8. Title: `India vs Germany | B.Tech | Full-Time | Indeed | Salary > $10,000`
9. Export: **task4_grouped_bar.png**

---

## 📊 Task 5 — Tree Map (Top 10 Companies)
**File to load:** `task5_top10.csv`

**Steps:**
1. In Marks panel → Change chart type to **Square** (Tree Map)
2. Drag **Company** → **Color** card
3. Drag **Postings** → **Size** card
4. Drag **Company** → **Label** card
5. Drag **Postings** → **Label** card (shows count inside each box)
6. Click **Label → Show Mark Labels**
7. Right-click any box → **Format** → Increase font size to 11
8. Title: `Top 10 Companies | Data Engineer/Data Scientist | LinkedIn | B.Tech | Female | Jan–Jun 2023`
9. Add annotation: `Visible: 3–5 PM IST only`
10. Export: **task5_treemap.png**

---

## 📊 Task 6 — Geographic Map with Drilldown
**File to load:** `task6_filtered.csv`

**Steps:**
1. Double-click **Latitude** in left panel → Tableau creates a map
2. Double-click **Longitude** → Adds to the map
3. In Marks panel → Change to **Circle**
4. Drag **Qualification** → **Color** card:
   - B.Tech = **Blue**
   - M.Tech = **Orange**
   - PhD = **Green**
5. Drag **Company Size** → **Size** card (bigger circles = bigger company)
6. Drag these to **Tooltip** card for drilldown:
   - Job Title, Company, Country, Salary, Company Size, Contact Person, Qualification
7. Go to **Map → Map Layers** → Enable **Country/Region borders**
8. Drag **Location** → **Detail** card (enables location drilldown on click)
9. Zoom map to Africa using Ctrl + Scroll
10. Title: `Qualification Map — African Countries | Full-Time | Indeed | Company Size > 80,000`
11. Add annotation: `Visible: 3–6 PM IST only`
12. Export: **task6_map.png**

---

## 🖥️ Final Dashboard in Tableau

1. Click **New Dashboard** tab at the bottom
2. Set size: **Dashboard → Fixed → 1200 x 900**
3. Drag all 6 task sheets onto the dashboard
4. Add title text box: `ElevanceSkills Internship Dashboard — Adinath Chavan`
5. Click any sheet → **More Options (▼) → Use as Filter** (makes charts interactive)
6. Save: **File → Save As → ElevanceSkills_Dashboard.twbx**

---

## ✅ Final Submission Checklist

| Item | Status |
|---|---|
| `ElevanceSkills_Internship.ipynb` | Jupyter Notebook (this file) |
| `task1_filtered.csv` to `task6_filtered.csv` | Filtered data files |
| `ElevanceSkills_Dashboard.twbx` | Tableau Dashboard |
| `internship_report.md` | Internship Report |
| Daily form filled every day | https://forms.gle/oi3UhE4RcZEKSu1r5 |

---
*Prepared by: Adinath Chavan | ElevanceSkills Technology Private Limited*

In [ ]:
print('=' * 55)
print('  ALL 6 TASKS COMPLETED!')
print('=' * 55)
print()
print('Python (NumPy + Pandas) Output Files:')
print('  ✅ task1_filtered.csv')
print('  ✅ task2_filtered.csv')
print('  ✅ task3_filtered.csv')
print('  ✅ task4_filtered.csv')
print('  ✅ task5_filtered.csv')
print('  ✅ task5_top10.csv')
print('  ✅ task6_filtered.csv')
print()
print('Next Step → Open each CSV in Tableau')
print('          → Follow the Tableau guide above')
print('          → Build all 6 charts')
print('          → Save as ElevanceSkills_Dashboard.twbx')
print('=' * 55)